# Insights Do Modelo — Previsão De Demanda

Este notebook apresenta os principais resultados do modelo Random Forest Regressor.

O objetivo é transformar as previsões em análises visuais para facilitar a interpretação do desempenho do modelo e apoiar a apresentação dos resultados.

In [0]:
import logging

In [0]:
%run /Workspace/Users/kalitamariano01@gmail.com/merca-data-platform/notebooks/utils/utils_feat_squad2_99_helpers

## Leitura Dos Resultados Do Modelo

Nesta etapa são lidas as tabelas geradas no notebook de treinamento:

- Resultado das previsões.
- Importância das features.

In [0]:
df_resultado_spark = spark.table("squad2.ml_resultado_previsao_demanda_rf_v1")
df_importancias_spark = spark.table("squad2.ml_importancia_features_previsao_demanda_rf_v1")

df_resultado = df_resultado_spark.toPandas()
df_importancias = df_importancias_spark.toPandas()

display(df_resultado_spark.limit(10))
display(df_importancias_spark)

## Comparação Entre Valor Real E Previsto

Este gráfico compara o total real de vendas do dia com o valor previsto pelo modelo.

Quanto mais próximos os pontos estiverem da linha diagonal, melhor está a previsão.

In [0]:
plt.figure(figsize=(8, 6))

plt.scatter(
    df_resultado["total_vendas_real"],
    df_resultado["total_vendas_previsto"],
    alpha=0.5
)

min_valor = min(df_resultado["total_vendas_real"].min(), df_resultado["total_vendas_previsto"].min())
max_valor = max(df_resultado["total_vendas_real"].max(), df_resultado["total_vendas_previsto"].max())

plt.plot([min_valor, max_valor], [min_valor, max_valor], color="red", linestyle="--")

plt.title("Previsão x Valor Real")
plt.xlabel("Total real de vendas do dia")
plt.ylabel("Total previsto pelo modelo")
plt.grid(True)
plt.show()

## Distribuição Do Erro Absoluto

Este gráfico mostra a distribuição dos erros do modelo.

Ele ajuda a identificar se a maior parte das previsões tem erro baixo ou se existem muitos casos com erro alto.

In [0]:
plt.figure(figsize=(8, 5))

plt.hist(df_resultado["erro_absoluto"], bins=30)

plt.title("Distribuição Do Erro Absoluto")
plt.xlabel("Erro absoluto")
plt.ylabel("Quantidade de previsões")
plt.grid(True)
plt.show()

## Erro Médio Por Hora Do Dia

Este gráfico mostra em quais horários o modelo tende a errar mais.

Isso é importante porque o problema de negócio envolve prever o fim do dia a partir da velocidade observada ao longo das horas.

In [0]:
erro_por_hora = (
    df_resultado
    .groupby("hora_evento", as_index=False)["erro_absoluto"]
    .mean()
)

plt.figure(figsize=(10, 5))

plt.plot(
    erro_por_hora["hora_evento"],
    erro_por_hora["erro_absoluto"],
    marker="o"
)

plt.title("Erro Médio Por Hora Do Dia")
plt.xlabel("Hora do dia")
plt.ylabel("Erro absoluto médio")
plt.grid(True)
plt.show()

## Comparação Entre Real E Previsto Por Hora

Este gráfico mostra como o modelo se comporta ao longo das horas do dia, comparando o total real de vendas com o total previsto.

Ele ajuda a identificar em quais horários o modelo acompanha bem a demanda e em quais horários começa a errar mais.

In [0]:
df_grafico = (
    df_resultado
    .groupby("hora_evento", as_index=False)
    .agg({
        "total_vendas_real": "mean",
        "total_vendas_previsto": "mean"
    })
    .sort_values("hora_evento")
)

plt.figure(figsize=(12, 5))

plt.plot(
    df_grafico["hora_evento"],
    df_grafico["total_vendas_real"],
    marker="o",
    label="Real"
)

plt.plot(
    df_grafico["hora_evento"],
    df_grafico["total_vendas_previsto"],
    marker="o",
    label="Previsto"
)

plt.title("Total Real x Previsto Por Hora")
plt.xlabel("Hora do dia")
plt.ylabel("Total de vendas do dia")
plt.legend()
plt.grid(True)
plt.show()

## Importância Das Features

Este gráfico mostra quais variáveis mais influenciaram o modelo.

Quanto maior a importância, mais aquela feature ajudou o Random Forest a prever o total de vendas do dia.

In [0]:
df_importancias_ordenado = df_importancias.sort_values("importancia", ascending=True)

plt.figure(figsize=(9, 5))

plt.barh(
    df_importancias_ordenado["feature"],
    df_importancias_ordenado["importancia"]
)

plt.title("Importância Das Features No Random Forest")
plt.xlabel("Importância")
plt.ylabel("Feature")
plt.grid(axis="x")
plt.show()

## Top 10 Maiores Erros

Esta tabela mostra os casos em que o modelo mais errou.

Ela ajuda a investigar possíveis dias atípicos, horários com poucos dados ou padrões difíceis de prever.

In [0]:
df_top_erros = (
    df_resultado
    .sort_values("erro_absoluto", ascending=False)
    .head(10)
)

display(spark.createDataFrame(df_top_erros))

## Resumo Dos Insights

Principais pontos observados:

- O modelo apresentou boa capacidade de previsão geral.
- O R² alto indica que o modelo capturou bem o padrão dos dados.
- O erro percentual médio ficou em torno de 10%.
- As features mais importantes indicam quais fatores mais influenciam a previsão de demanda.
- Os maiores erros devem ser analisados para identificar dias com comportamento atípico ou poucas janelas de venda.

Esses insights podem ser usados no dashboard para explicar a qualidade do modelo e apoiar decisões operacionais.